# AgeLens — 12 Baseline Characteristics

## Purpose

This notebook creates publication-ready, survey-weighted descriptive baseline tables from the governed AgeLens V1 mortality cohort.

It produces:

- a compact main-article table containing demographics and Phenotypic Age measures;
- a supplementary table containing the nine harmonized biomarkers;
- numeric long-form outputs for auditability;
- validation checks and metadata.

No hypothesis tests or p-values are generated. The tables are descriptive and do not alter any governed scientific result.


In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_FOLDER_NAME = "nhanes"
NOTEBOOK_BUILD = "12-v1-baseline-characteristics"
EXPECTED_COHORT_N = 4350
EXPECTED_DEATHS = 127

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 220)

print(f"Notebook build: {NOTEBOOK_BUILD}")
print(f"numpy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"Current working directory: {Path.cwd().resolve()}")


In [ ]:
def find_project_root(folder_name: str = PROJECT_FOLDER_NAME) -> Path:
    current = Path.cwd().resolve()

    for candidate in [current, *current.parents]:
        if candidate.name.lower() == folder_name.lower():
            return candidate

    raise FileNotFoundError(
        f"Could not find a parent folder named '{folder_name}'. "
        "Run this notebook from inside the nhanes project."
    )


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "configs" / "agelens_config.json"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(CONFIG_PATH)

CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

PROCESSED_ROOT = PROJECT_ROOT / CONFIG["paths"]["processed_data"]
INTERIM_ROOT = PROJECT_ROOT / CONFIG["paths"]["interim_data"]
TABLES_ROOT = PROJECT_ROOT / CONFIG["paths"]["tables"]
LOGS_ROOT = PROJECT_ROOT / CONFIG["paths"]["logs"]
DOCS_RESULTS_ROOT = PROJECT_ROOT / "docs" / "results"

for path in [TABLES_ROOT, LOGS_ROOT, DOCS_RESULTS_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

COHORT_PATH = (
    PROCESSED_ROOT
    / "agelens_v1_mortality_cohort_authorized.parquet"
)
PREPROCESSED_PATH = (
    INTERIM_ROOT
    / "nhanes_2015_2018_preprocessed_diagnostic.parquet"
)

required_paths = [COHORT_PATH, PREPROCESSED_PATH]
missing_paths = [path for path in required_paths if not path.exists()]

if missing_paths:
    raise FileNotFoundError(
        "Required participant-level governed inputs are missing: "
        f"{missing_paths}"
    )

print(f"Project root: {PROJECT_ROOT}")
print(f"Mortality cohort: {COHORT_PATH}")
print(f"Preprocessed biomarkers: {PREPROCESSED_PATH}")


In [ ]:
cohort = pd.read_parquet(COHORT_PATH)
preprocessed = pd.read_parquet(PREPROCESSED_PATH)

cohort_required = {
    "SEQN",
    "NHANES_CYCLE",
    "chronological_age_years",
    "age_topcoded",
    "RIAGENDR",
    "RIDRETH3",
    "WTSAF4YR",
    "phenotypic_age_years",
    "phenoage_acceleration_years",
    "mortality_event",
}

biomarker_columns = {
    "albumin_harmonized_g_dL": "Albumin, g/dL",
    "creatinine_harmonized_mg_dL": "Creatinine, mg/dL",
    "glucose_raw_mg_dL": "Fasting glucose, mg/dL",
    "crp_harmonized_mg_L": "hs-CRP, mg/L",
    "lymphocyte_percent": "Lymphocyte percentage, %",
    "mcv_fL": "Mean cell volume, fL",
    "rdw_percent": "Red cell distribution width, %",
    "alp_harmonized_U_L": "Alkaline phosphatase, U/L",
    "wbc_1000cells_uL": "White blood cell count, 10³ cells/µL",
}

preprocessed_required = {
    "SEQN",
    "NHANES_CYCLE",
    *biomarker_columns.keys(),
}

missing_cohort = sorted(cohort_required - set(cohort.columns))
missing_preprocessed = sorted(preprocessed_required - set(preprocessed.columns))

if missing_cohort:
    raise ValueError(f"Cohort is missing columns: {missing_cohort}")

if missing_preprocessed:
    raise ValueError(
        f"Preprocessed file is missing columns: {missing_preprocessed}"
    )

for frame_name, frame in {
    "cohort": cohort,
    "preprocessed": preprocessed,
}.items():
    frame["SEQN"] = pd.to_numeric(
        frame["SEQN"],
        errors="raise",
    ).astype("int64")

    if frame.duplicated(["NHANES_CYCLE", "SEQN"]).any():
        raise RuntimeError(
            f"Duplicate cycle + SEQN rows in {frame_name}."
        )

biomarkers = preprocessed.loc[
    :,
    [
        "SEQN",
        "NHANES_CYCLE",
        *biomarker_columns.keys(),
    ],
].copy()

analysis = cohort.merge(
    biomarkers,
    on=["SEQN", "NHANES_CYCLE"],
    how="left",
    validate="one_to_one",
)

if len(analysis) != EXPECTED_COHORT_N:
    raise RuntimeError(
        f"Unexpected cohort size: {len(analysis)}; "
        f"expected {EXPECTED_COHORT_N}."
    )

if int(analysis["mortality_event"].sum()) != EXPECTED_DEATHS:
    raise RuntimeError(
        "Unexpected death count: "
        f"{int(analysis['mortality_event'].sum())}; "
        f"expected {EXPECTED_DEATHS}."
    )

if analysis[list(biomarker_columns)].isna().any().any():
    missing_counts = (
        analysis[list(biomarker_columns)]
        .isna()
        .sum()
    )
    raise RuntimeError(
        "Governed mortality cohort contains missing baseline "
        f"biomarkers after merge: {missing_counts[missing_counts > 0].to_dict()}"
    )

print(f"Governed mortality cohort rows: {len(analysis):,}")
print(f"Observed deaths: {int(analysis['mortality_event'].sum()):,}")
display(
    analysis.groupby("NHANES_CYCLE", observed=True).agg(
        n=("SEQN", "size"),
        deaths=("mortality_event", "sum"),
        weighted_population=("WTSAF4YR", "sum"),
    )
)


In [ ]:
GROUPS = {
    "Overall": pd.Series(True, index=analysis.index),
    "2015–2016": analysis["NHANES_CYCLE"].eq("2015_2016"),
    "2017–2018": analysis["NHANES_CYCLE"].eq("2017_2018"),
}

RACE_LABELS = {
    1: "Mexican American, %",
    2: "Other Hispanic, %",
    3: "Non-Hispanic White, %",
    4: "Non-Hispanic Black, %",
    6: "Non-Hispanic Asian, %",
    7: "Other or multiracial, %",
}


def valid_weighted_frame(
    frame: pd.DataFrame,
    value_column: str,
) -> pd.DataFrame:
    valid = frame.loc[
        :,
        [value_column, "WTSAF4YR"],
    ].dropna()

    valid = valid.loc[
        valid["WTSAF4YR"].gt(0)
    ]

    if valid.empty:
        raise RuntimeError(
            f"No valid weighted observations for {value_column}."
        )

    return valid


def weighted_mean_sd(
    frame: pd.DataFrame,
    value_column: str,
) -> tuple[float, float, int]:
    valid = valid_weighted_frame(
        frame,
        value_column,
    )

    values = valid[value_column].to_numpy(dtype=float)
    weights = valid["WTSAF4YR"].to_numpy(dtype=float)

    mean = float(
        np.average(values, weights=weights)
    )
    variance = float(
        np.average(
            np.square(values - mean),
            weights=weights,
        )
    )

    return mean, float(np.sqrt(variance)), int(len(valid))


def weighted_percent(
    frame: pd.DataFrame,
    indicator: pd.Series,
) -> tuple[float, int]:
    valid = frame.loc[
        :,
        ["WTSAF4YR"],
    ].copy()
    valid["indicator"] = (
        indicator.reindex(frame.index)
        .astype("float64")
    )
    valid = valid.dropna()
    valid = valid.loc[
        valid["WTSAF4YR"].gt(0)
    ]

    percent = 100.0 * float(
        np.average(
            valid["indicator"],
            weights=valid["WTSAF4YR"],
        )
    )

    return percent, int(valid["indicator"].sum())


def format_mean_sd(mean: float, sd: float) -> str:
    return f"{mean:.2f} ({sd:.2f})"


def format_percent(percent: float) -> str:
    return f"{percent:.1f}%"


def build_continuous_row(
    label: str,
    column: str,
) -> tuple[dict[str, str], list[dict[str, object]]]:
    formatted = {"Characteristic": label}
    numeric_records = []

    for group_label, mask in GROUPS.items():
        frame = analysis.loc[mask]
        mean, sd, n = weighted_mean_sd(
            frame,
            column,
        )
        formatted[group_label] = format_mean_sd(
            mean,
            sd,
        )
        numeric_records.append(
            {
                "table": "main",
                "group": group_label,
                "characteristic": label,
                "statistic": "weighted_mean_sd",
                "n": n,
                "weighted_mean": mean,
                "weighted_sd": sd,
                "weighted_percent": np.nan,
            }
        )

    return formatted, numeric_records


def build_percent_row(
    label: str,
    indicator_builder,
) -> tuple[dict[str, str], list[dict[str, object]]]:
    formatted = {"Characteristic": label}
    numeric_records = []

    for group_label, mask in GROUPS.items():
        frame = analysis.loc[mask]
        indicator = indicator_builder(frame)
        percent, unweighted_yes = weighted_percent(
            frame,
            indicator,
        )
        formatted[group_label] = format_percent(
            percent,
        )
        numeric_records.append(
            {
                "table": "main",
                "group": group_label,
                "characteristic": label,
                "statistic": "weighted_percent",
                "n": int(len(frame)),
                "unweighted_yes": unweighted_yes,
                "weighted_mean": np.nan,
                "weighted_sd": np.nan,
                "weighted_percent": percent,
            }
        )

    return formatted, numeric_records


In [ ]:
main_rows = []
numeric_records = []

n_row = {"Characteristic": "Unweighted n"}
for group_label, mask in GROUPS.items():
    n_row[group_label] = f"{int(mask.sum()):,}"
main_rows.append(n_row)

for label, column in [
    ("Chronological age, years", "chronological_age_years"),
    ("Phenotypic Age, years", "phenotypic_age_years"),
    ("Phenotypic Age acceleration, years", "phenoage_acceleration_years"),
]:
    row, records = build_continuous_row(
        label,
        column,
    )
    main_rows.append(row)
    numeric_records.extend(records)

for label, builder in [
    ("Female, %", lambda frame: frame["RIAGENDR"].eq(2)),
    (
        "Age top-coded at 80 years, %",
        lambda frame: frame["age_topcoded"].astype(bool),
    ),
]:
    row, records = build_percent_row(
        label,
        builder,
    )
    main_rows.append(row)
    numeric_records.extend(records)

for code, label in RACE_LABELS.items():
    row, records = build_percent_row(
        label,
        lambda frame, code=code: frame["RIDRETH3"].eq(code),
    )
    main_rows.append(row)
    numeric_records.extend(records)

main_table = pd.DataFrame(main_rows)

biomarker_rows = []
biomarker_numeric_records = []

for column, label in biomarker_columns.items():
    formatted = {"Biomarker": label}

    for group_label, mask in GROUPS.items():
        frame = analysis.loc[mask]
        mean, sd, n = weighted_mean_sd(
            frame,
            column,
        )
        formatted[group_label] = format_mean_sd(
            mean,
            sd,
        )
        biomarker_numeric_records.append(
            {
                "table": "biomarkers",
                "group": group_label,
                "characteristic": label,
                "statistic": "weighted_mean_sd",
                "n": n,
                "weighted_mean": mean,
                "weighted_sd": sd,
                "weighted_percent": np.nan,
            }
        )

    biomarker_rows.append(formatted)

biomarker_table = pd.DataFrame(
    biomarker_rows
)
numeric_long = pd.DataFrame(
    numeric_records
    + biomarker_numeric_records
)

display(main_table)
display(biomarker_table)


In [ ]:
validation_records = []

cycle_counts = (
    analysis.groupby(
        "NHANES_CYCLE",
        observed=True,
    )
    .size()
    .to_dict()
)

validation_records.extend(
    [
        {
            "check": "Mortality cohort n equals governed value",
            "pass": len(analysis) == EXPECTED_COHORT_N,
            "observed": len(analysis),
        },
        {
            "check": "Death count equals governed value",
            "pass": int(analysis["mortality_event"].sum()) == EXPECTED_DEATHS,
            "observed": int(analysis["mortality_event"].sum()),
        },
        {
            "check": "2015–2016 cohort n equals governed value",
            "pass": cycle_counts.get("2015_2016") == 2178,
            "observed": cycle_counts.get("2015_2016"),
        },
        {
            "check": "2017–2018 cohort n equals governed value",
            "pass": cycle_counts.get("2017_2018") == 2172,
            "observed": cycle_counts.get("2017_2018"),
        },
        {
            "check": "No baseline biomarker is missing",
            "pass": not analysis[list(biomarker_columns)].isna().any().any(),
            "observed": int(
                analysis[list(biomarker_columns)]
                .isna()
                .sum()
                .sum()
            ),
        },
        {
            "check": "Sex codes are restricted to 1 and 2",
            "pass": set(
                analysis["RIAGENDR"].dropna().astype(int).unique()
            ).issubset({1, 2}),
            "observed": sorted(
                analysis["RIAGENDR"].dropna().astype(int).unique().tolist()
            ),
        },
        {
            "check": "Race/ethnicity codes are recognized",
            "pass": set(
                analysis["RIDRETH3"].dropna().astype(int).unique()
            ).issubset(set(RACE_LABELS)),
            "observed": sorted(
                analysis["RIDRETH3"].dropna().astype(int).unique().tolist()
            ),
        },
        {
            "check": "No p-value column was created",
            "pass": not any(
                "p" == str(column).lower()
                or "p_value" in str(column).lower()
                or "p-value" in str(column).lower()
                for column in [
                    *main_table.columns,
                    *biomarker_table.columns,
                    *numeric_long.columns,
                ]
            ),
            "observed": True,
        },
    ]
)

for group_label, mask in GROUPS.items():
    frame = analysis.loc[mask]
    race_sum = sum(
        weighted_percent(
            frame,
            frame["RIDRETH3"].eq(code),
        )[0]
        for code in RACE_LABELS
    )
    validation_records.append(
        {
            "check": f"Race percentages sum to 100 in {group_label}",
            "pass": abs(race_sum - 100.0) < 1e-8,
            "observed": race_sum,
        }
    )

validation = pd.DataFrame(
    validation_records
)

if not validation["pass"].all():
    display(
        validation.loc[
            ~validation["pass"]
        ]
    )
    raise RuntimeError(
        "Baseline table validation failed."
    )

display(validation)
print("✅ Baseline table validation passed.")


In [ ]:
MAIN_FORMATTED_PATH = (
    TABLES_ROOT
    / "12_baseline_characteristics_main_formatted.csv"
)
BIOMARKER_FORMATTED_PATH = (
    TABLES_ROOT
    / "12_baseline_characteristics_biomarkers_formatted.csv"
)
NUMERIC_LONG_PATH = (
    TABLES_ROOT
    / "12_baseline_characteristics_numeric_long.csv"
)
VALIDATION_PATH = (
    TABLES_ROOT
    / "12_baseline_characteristics_checks.csv"
)
MARKDOWN_PATH = (
    DOCS_RESULTS_ROOT
    / "12_Baseline_Characteristics.md"
)
METADATA_PATH = (
    LOGS_ROOT
    / "12_baseline_characteristics_metadata.json"
)

main_table.to_csv(
    MAIN_FORMATTED_PATH,
    index=False,
)
biomarker_table.to_csv(
    BIOMARKER_FORMATTED_PATH,
    index=False,
)
numeric_long.to_csv(
    NUMERIC_LONG_PATH,
    index=False,
)
validation.to_csv(
    VALIDATION_PATH,
    index=False,
)

def dataframe_to_markdown(frame: pd.DataFrame) -> str:
    headers = [str(column) for column in frame.columns]
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]

    for row in frame.itertuples(index=False, name=None):
        values = [
            str(value).replace("|", "\\|")
            for value in row
        ]
        lines.append(
            "| " + " | ".join(values) + " |"
        )

    return "\n".join(lines)


markdown = f"""# AgeLens V1 Baseline Characteristics

## Main article table

Values are survey-weighted mean (descriptive weighted standard deviation) or survey-weighted percentage unless otherwise stated.

{dataframe_to_markdown(main_table)}

## Supplementary biomarker table

Values are survey-weighted mean (descriptive weighted standard deviation).

{dataframe_to_markdown(biomarker_table)}

## Table note

The analytic population is the governed AgeLens V1 all-cause mortality cohort. Estimates use the pooled fasting-subsample weight `WTSAF4YR`. Columns are descriptive; no between-cycle hypothesis tests or p-values were calculated. Chronological age is top-coded at 80 years in public-use NHANES. Biomarkers reflect the governed harmonization and unit policy used for the canonical Phenotypic Age computation.
"""

MARKDOWN_PATH.write_text(
    markdown,
    encoding="utf-8",
)

metadata = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "notebook": "12_baseline_characteristics.ipynb",
    "notebook_build": NOTEBOOK_BUILD,
    "cohort_file": str(
        COHORT_PATH.relative_to(PROJECT_ROOT)
    ),
    "preprocessed_file": str(
        PREPROCESSED_PATH.relative_to(PROJECT_ROOT)
    ),
    "cohort_n": int(len(analysis)),
    "deaths": int(
        analysis["mortality_event"].sum()
    ),
    "groups": list(GROUPS),
    "weight": "WTSAF4YR",
    "statistics": [
        "survey-weighted mean",
        "descriptive weighted standard deviation",
        "survey-weighted percentage",
    ],
    "hypothesis_tests_run": False,
    "p_values_generated": False,
    "outputs": [
        str(path.relative_to(PROJECT_ROOT))
        for path in [
            MAIN_FORMATTED_PATH,
            BIOMARKER_FORMATTED_PATH,
            NUMERIC_LONG_PATH,
            VALIDATION_PATH,
            MARKDOWN_PATH,
        ]
    ],
}

METADATA_PATH.write_text(
    json.dumps(
        metadata,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("Files written:")
for path in [
    MAIN_FORMATTED_PATH,
    BIOMARKER_FORMATTED_PATH,
    NUMERIC_LONG_PATH,
    VALIDATION_PATH,
    MARKDOWN_PATH,
    METADATA_PATH,
]:
    print(f"  - {path.relative_to(PROJECT_ROOT)}")

print("✅ Publication baseline tables completed.")
